⚠️ This notebook is for experimentation and analysis only.
Production logic is implemented under `src/` and `pipeline/`.


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path


In [2]:
INPUT_PATH = "../data/processed/train_fe.csv"
OUTPUT_DIR = Path("../data/processed/baselines")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])


In [3]:
df = df.sort_values(["item_id", "date"]).reset_index(drop=True)


In [4]:
HORIZON = 28

train_df = (
    df
    .groupby("item_id")
    .apply(lambda x: x.iloc[:-HORIZON])
    .reset_index(drop=True)
)

valid_df = (
    df
    .groupby("item_id")
    .apply(lambda x: x.iloc[-HORIZON:])
    .reset_index(drop=True)
)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14476\2039935084.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:-HORIZON])
C:\Users\ASUS\AppData\Local\Temp\ipykernel_14476\2039935084.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[-HORIZON:])


In [5]:
last_sales = (
    train_df
    .groupby("item_id")["sales"]
    .last()
)

valid_df["pred_naive"] = valid_df["item_id"].map(last_sales)


In [6]:
ma7 = (
    train_df
    .groupby("item_id")["sales"]
    .apply(lambda x: x.tail(7).mean())
)

valid_df["pred_ma7"] = valid_df["item_id"].map(ma7)


In [7]:
ma28 = (
    train_df
    .groupby("item_id")["sales"]
    .apply(lambda x: x.tail(28).mean())
)

valid_df["pred_ma28"] = valid_df["item_id"].map(ma28)


In [8]:
pred_cols = [
    "date", "store_id", "item_id", "sales",
    "pred_naive", "pred_ma7", "pred_ma28"
]

baseline_predictions = valid_df[pred_cols]

baseline_predictions.to_csv(
    OUTPUT_DIR / "baseline_predictions.csv",
    index=False
)


In [9]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }



In [10]:
metrics = {
    "Naive": evaluate(valid_df["sales"], valid_df["pred_naive"]),
    "MA_7": evaluate(valid_df["sales"], valid_df["pred_ma7"]),
    "MA_28": evaluate(valid_df["sales"], valid_df["pred_ma28"]),
}


In [11]:
print(metrics)

{'Naive': {'MAE': 1.485194208874104, 'RMSE': np.float64(3.953843356783116)}, 'MA_7': {'MAE': 1.1979203619788354, 'RMSE': np.float64(2.8719536003354995)}, 'MA_28': {'MAE': 1.198481770537011, 'RMSE': np.float64(2.669883926726821)}}


In [12]:
baseline_metrics = (
    pd.DataFrame(metrics)
    .T
    .reset_index()
    .rename(columns={"index": "model"})
)

baseline_metrics.to_csv(
    OUTPUT_DIR / "baseline_metrics.csv",
    index=False
)


## Baseline Model Results

Three baseline forecasting models were evaluated at the item level using a
28-day forecasting horizon.

| Model  | MAE  | RMSE |
|-------|------|------|
| Naive | 1.49 | 3.95 |
| MA-7  | 1.20 | 2.87 |
| MA-28 | 1.20 | 2.67 |

Both moving average models substantially outperform the naive baseline,
highlighting the importance of incorporating historical context when forecasting
intermittent retail demand.  

The 7-day and 28-day moving averages achieve very similar MAE values, while the
28-day moving average yields the lowest RMSE, indicating improved stability and
error reduction when longer historical windows are used.

These results establish a strong baseline and demonstrate that simple temporal
smoothing techniques can capture a significant portion of the demand signal.
They also motivate the use of machine learning models to further model
non-linear patterns, longer-term dependencies, and external effects such as
events and store-level heterogeneity.
